# E-commerce Orders — Streaming Cleaning & KPI Pipeline

**Session 04 — streaming exercise**
Author: Khushbu Shobhashna

**Dataset:** UCI *Online Retail* (Kaggle: `carrie1/ecommerce-data`, file `data.csv`) — ~541k real
transactions from a UK online gift retailer, Dec 2010 – Dec 2011.

---

### Why this dataset

The dirt here is **real**, not injected. Nobody wrote `"ERROR"` into a cell — these are the
artifacts of an actual operational system, which is what you meet in production:

| Pattern | What it really is | Why it matters |
|---|---|---|
| `InvoiceNo` starting with `C` | a cancellation | double-counts revenue if not excluded |
| `Quantity` negative | a return | legitimate data, **not** an error to drop |
| `CustomerID` null (~25%) | guest / unlinked checkout | silently breaks customer-level KPIs |
| `UnitPrice` = 0 or negative | samples, adjustments, bad debt | skews AOV badly |
| `StockCode` like `POST`, `D`, `M`, `BANK CHARGES` | not products at all | pollutes "top products" |
| `Description` null / whitespace | missing catalogue join | — |
| exact duplicate rows | double-submitted lines | inflates every sum |
| file encoding `ISO-8859-1` | not UTF-8 | mis-reads silently corrupt product names |

The important judgement call: **a return is not dirty data.** Most of these need
*classifying*, not deleting — so the pipeline tags rows and lets each KPI decide.

---

### Architecture — medallion, with Auto Loader

```
   CSV chunks landing in a UC Volume
              |
              |  readStream + Auto Loader (cloudFiles)
              v
        BRONZE  raw_orders          all strings, nothing thrown away
              |
              |  clean_orders(df)   <- ONE function, batch and stream both
              v
        SILVER  cleaned_orders      typed, classified, quality-flagged
              |
              v
        GOLD    KPI tables
```

The point of the exercise: `clean_orders()` is a plain DataFrame → DataFrame function.
Spark runs the *identical* code batch or streaming — that engine-agnosticism is the
real lesson of Structured Streaming.

## 0. Imports and configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [0]:
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA  = "retail_streaming"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.landing")

VOLUME    = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
INBOX     = f"{VOLUME}/incoming"        # Auto Loader watches this directory
SCHEMA_LOC = f"{VOLUME}/_schema"        # Auto Loader schema tracking
CKPT_BRONZE = f"{VOLUME}/_ckpt_bronze"
CKPT_SILVER = f"{VOLUME}/_ckpt_silver"

dbutils.fs.mkdirs(INBOX)

print(f"Catalog/schema : {CATALOG}.{SCHEMA}")
print(f"Landing volume : {VOLUME}")

## 1. Land the source data

Put `data.csv` (from the Kaggle dataset) **next to this notebook**. The cell below copies it into
the volume and splits it into chunks, so files arrive incrementally and Auto Loader has something
to actually stream.

If the CSV is absent, a synthetic generator produces a file with the *same schema and the same
dirt patterns* so the notebook still runs end to end. It is clearly labelled — swap in the real
file when you have it.

In [0]:
import os, csv, random, shutil, datetime

SOURCE_CSV = os.path.join(os.getcwd(), "data.csv")
USING_SYNTHETIC = not os.path.exists(SOURCE_CSV)

HEADER = ["InvoiceNo", "StockCode", "Description", "Quantity",
          "InvoiceDate", "UnitPrice", "CustomerID", "Country"]

if USING_SYNTHETIC:
    print("!! data.csv not found — generating SYNTHETIC data with the same dirt patterns.")
    print("!! Download carrie1/ecommerce-data from Kaggle and re-run for the real thing.\n")

    random.seed(42)   # reproducible
    products = [
        ("85123A", "WHITE HANGING HEART T-LIGHT HOLDER", 2.55),
        ("71053",  "WHITE METAL LANTERN", 3.39),
        ("84406B", "CREAM CUPID HEARTS COAT HANGER", 2.75),
        ("22423",  "REGENCY CAKESTAND 3 TIER", 12.75),
        ("47566",  "PARTY BUNTING", 4.95),
        ("20725",  "LUNCH BAG RED RETROSPOT", 1.65),
        ("22086",  "PAPER CHAIN KIT 50'S CHRISTMAS", 2.95),
        ("23084",  "RABBIT NIGHT LIGHT", 2.08),
    ]
    non_products = [("POST", "POSTAGE", 18.0), ("D", "Discount", -11.0),
                    ("M", "Manual", 1.25), ("BANK CHARGES", "Bank Charges", 15.0)]
    countries = (["United Kingdom"] * 12 + ["Germany", "France", "EIRE",
                 "Spain", "Netherlands", "Belgium", "Unspecified"])

    rows, invoice_no = [], 536365
    start = datetime.date(2010, 12, 1)

    for _ in range(6000):
        invoice_no += 1
        is_cancel = random.random() < 0.017
        inv = f"C{invoice_no}" if is_cancel else str(invoice_no)
        cust = "" if random.random() < 0.25 else str(random.randint(12346, 18287))
        country = random.choice(countries)
        day = start + datetime.timedelta(days=random.randint(0, 364))
        ts = f"{day.month}/{day.day}/{day.year} {random.randint(6,20)}:{random.randint(0,59):02d}"

        for _ in range(random.randint(1, 5)):
            if random.random() < 0.03:
                code_, desc, price = random.choice(non_products)
            else:
                code_, desc, price = random.choice(products)
            qty = -random.randint(1, 6) if is_cancel else random.randint(1, 24)
            if random.random() < 0.02:
                desc = ""                       # missing description
            if random.random() < 0.015:
                price = 0.0                     # zero price
            rows.append([inv, code_, desc, qty, ts,
                         f"{price:.2f}", cust, country])

    dupes = [list(r) for r in random.sample(rows, k=int(len(rows) * 0.01))]
    rows.extend(dupes)                          # exact duplicate rows
    random.shuffle(rows)

    SOURCE_CSV = "/tmp/data_synthetic.csv"
    with open(SOURCE_CSV, "w", newline="", encoding="ISO-8859-1") as f:
        w = csv.writer(f)
        w.writerow(HEADER)
        w.writerows(rows)
    print(f"Generated {len(rows):,} rows -> {SOURCE_CSV}")
else:
    print(f"Using real dataset: {SOURCE_CSV}")

### Split into chunks to simulate incremental arrival

A single file is a batch wearing a streaming costume. Splitting it means Auto Loader genuinely
discovers new files across runs, which is what lets you *prove* the pipeline is incremental
in section 7.

In [0]:
N_CHUNKS = 4
HOLDBACK_DIR = f"{VOLUME}/_holdback"
HOLDBACK_CSV = f"{HOLDBACK_DIR}/orders_part_{N_CHUNKS - 1:02d}.csv"

# Start clean so re-running the notebook is deterministic.
for p in (INBOX, SCHEMA_LOC, CKPT_BRONZE, CKPT_SILVER, HOLDBACK_DIR):
    try:
        dbutils.fs.rm(p, recurse=True)
    except Exception:
        pass
dbutils.fs.mkdirs(INBOX)
dbutils.fs.mkdirs(HOLDBACK_DIR)

with open(SOURCE_CSV, "r", encoding="ISO-8859-1") as f:
      header, body = f.readline(), f.readlines()

chunk_size = len(body) // N_CHUNKS + 1
for i in range(N_CHUNKS):
      part = body[i * chunk_size : (i + 1) * chunk_size]
      if not part:
          continue
      # Hold the last chunk back — section 6 drops it in to demonstrate incrementality.
      # Everything is written straight into the volume: serverless has no writable local FS.
      target = HOLDBACK_CSV if i == N_CHUNKS - 1 else f"{INBOX}/orders_part_{i:02d}.csv"
      with open(target, "w", newline="", encoding="ISO-8859-1") as out:
          out.write(header)
          out.writelines(part)

print(f"Landed {N_CHUNKS - 1} chunks in {INBOX}")
print(f"Held back for section 6: {HOLDBACK_CSV}")
display(dbutils.fs.ls(INBOX))


## 2. Profile before cleaning

Never clean blind. This is a small batch read purely to *see* the dirt — it does not feed the
streaming pipeline.

In [0]:
sample = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("encoding", "ISO-8859-1")     # NOT UTF-8 — this dataset will mis-read without it
    .csv(f"{INBOX}/*.csv")
)

print(f"Sample rows: {sample.count():,}")
display(sample.limit(15))

In [0]:
checks = sample.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("CustomerID").isNull().cast("int")).alias("null_customer_id"),
    F.sum(F.col("Description").isNull().cast("int")).alias("null_description"),
    F.sum((F.col("Quantity").cast("double") < 0).cast("int")).alias("negative_quantity"),
    F.sum((F.col("UnitPrice").cast("double") <= 0).cast("int")).alias("non_positive_price"),
    F.sum(F.col("InvoiceNo").startswith("C").cast("int")).alias("cancellations"),
)
display(checks)

print("Duplicate rows:", sample.count() - sample.dropDuplicates().count())
print("\nNon-product StockCodes present:")
display(
    sample.filter(F.col("StockCode").rlike("^[A-Za-z ]+$"))
    .groupBy("StockCode", "Description").count()
    .orderBy(F.desc("count"))
)

## 3. The cleaning transform — one function, both engines

Everything below is a **narrow transformation**: no shuffles, no aggregations, no actions. That
is what makes it legal in a streaming query, and it is why the exact same function can be handed
a batch DataFrame or a streaming one.

Design decisions worth defending in review:

- **Nothing is dropped.** Returns and cancellations are *classified*, because "revenue excluding
  returns" and "return rate" are both real KPIs and each needs different rows.
- `line_revenue = Quantity x UnitPrice` is computed once here, not in eight different KPI cells.
- Non-product stock codes (`POST`, `D`, `M`, …) are flagged, not deleted — postage is real money,
  it just is not a product.
- Every quality problem becomes a boolean column, so the data-quality KPI is a sum of flags
  rather than a second pass over the data.

In [0]:
NON_PRODUCT_CODES = ["POST", "D", "M", "C2", "DOT", "CRUK", "PADS",
                     "BANK CHARGES", "AMAZONFEE", "S", "B"]

def clean_orders(df):
    """Bronze -> Silver. Pure narrow transform: identical under batch and streaming."""

    # --- normalise blanks/sentinels to real NULLs, uniformly -----------------
    for c in ["InvoiceNo", "StockCode", "Description", "CustomerID", "Country"]:
        df = df.withColumn(
            c,
            F.when(
                F.col(c).isNull() | (F.trim(F.col(c)) == "") | (F.upper(F.trim(F.col(c))) == "UNSPECIFIED"),
                F.lit(None).cast("string"),
            ).otherwise(F.trim(F.col(c))),
        )

    # --- typing --------------------------------------------------------------
    df = (
        df
        .withColumn("Quantity", F.col("Quantity").cast("int"))
        .withColumn("UnitPrice", F.col("UnitPrice").cast("double"))
        .withColumn("CustomerID", F.col("CustomerID").cast("int"))
        # Source format is M/d/yyyy H:mm — 12/1/2010 8:26. Not ISO.
        .withColumn("InvoiceTimestamp", F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
    )

    # --- classification ------------------------------------------------------
    df = (
        df
        .withColumn("is_cancellation", F.coalesce(F.col("InvoiceNo").startswith("C"), F.lit(False)))
        .withColumn("is_return", F.col("Quantity") < 0)
        .withColumn("is_non_product", F.upper(F.col("StockCode")).isin(NON_PRODUCT_CODES))
        .withColumn("is_guest_checkout", F.col("CustomerID").isNull())
    )

    # --- quality flags -------------------------------------------------------
    df = (
        df
        .withColumn("has_invalid_price", F.col("UnitPrice").isNull() | (F.col("UnitPrice") <= 0))
        .withColumn("has_missing_description", F.col("Description").isNull())
        .withColumn("has_unparseable_date",
                    F.col("InvoiceDate").isNotNull() & F.col("InvoiceTimestamp").isNull())
    )

    # --- derived measures ----------------------------------------------------
    df = (
        df
        .withColumn("line_revenue", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
        .withColumn("invoice_date", F.to_date("InvoiceTimestamp"))
        .withColumn("year_month", F.date_format("InvoiceTimestamp", "yyyy-MM"))
        .withColumn("invoice_hour", F.hour("InvoiceTimestamp"))
    )

    # A line is analysable when it can contribute to revenue reporting.
    df = df.withColumn(
        "is_valid_sale",
        ~F.col("is_cancellation")
        & ~F.col("is_return")
        & ~F.col("has_invalid_price")
        & F.col("Quantity").isNotNull()
        & (F.col("Quantity") > 0)
        & F.col("InvoiceTimestamp").isNotNull(),
    )

    return df.withColumn("ingested_at", F.current_timestamp())

## 4. Bronze — Auto Loader ingestion

An explicit schema (all strings) is supplied rather than letting Auto Loader infer. Inference on
dirty CSV guesses types from a sample, then chokes when row 400,000 disagrees — and the rescued-data
column becomes a dumping ground. Reading as string and casting deliberately in section 3 is the
more predictable contract.

`trigger(availableNow=True)` processes every file waiting right now, then stops. It's the right
trigger for a notebook: real incremental semantics, no cluster left spinning.

In [0]:
bronze_schema = T.StructType([T.StructField(c, T.StringType(), True) for c in
    ["InvoiceNo", "StockCode", "Description", "Quantity",
     "InvoiceDate", "UnitPrice", "CustomerID", "Country"]])

bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)
    .option("header", True)
    .option("encoding", "ISO-8859-1")
    .schema(bronze_schema)
    .load(INBOX)
    .withColumn("source_file", F.col("_metadata.file_path"))
)

print("Is this a streaming DataFrame?", bronze_stream.isStreaming)

q_bronze = (
    bronze_stream.writeStream
    .option("checkpointLocation", CKPT_BRONZE)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("raw_orders")
)
q_bronze.awaitTermination()

print(f"Bronze rows: {spark.table('raw_orders').count():,}")
display(spark.table("raw_orders").limit(5))

## 5. Silver — the same function, now on a stream

`clean_orders` is called here with a *streaming* DataFrame. Not one line of it changes. If you
had used a `.count()` or a `.collect()` inside it, this cell would fail — that constraint is
exactly what keeps transform logic portable.

In [0]:
silver_stream = clean_orders(spark.readStream.table("raw_orders"))

print("Is this a streaming DataFrame?", silver_stream.isStreaming)

q_silver = (
    silver_stream.writeStream
    .option("checkpointLocation", CKPT_SILVER)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("cleaned_orders")
)
q_silver.awaitTermination()

silver = spark.table("cleaned_orders")
print(f"Silver rows: {silver.count():,}")
display(silver.limit(10))

### Batch/stream parity check

The claim "the same code runs on both engines" is worth *verifying*, not asserting. Running
`clean_orders` in batch over the same input should produce identical row counts and revenue.

In [0]:
batch_equivalent = clean_orders(
    spark.read.option("header", True).option("inferSchema", False)
    .option("encoding", "ISO-8859-1").csv(f"{INBOX}/*.csv")
)

stream_rows = silver.count()
batch_rows  = batch_equivalent.count()
stream_rev  = silver.agg(F.round(F.sum("line_revenue"), 2)).first()[0]
batch_rev   = batch_equivalent.agg(F.round(F.sum("line_revenue"), 2)).first()[0]

display(spark.createDataFrame(
    [("rows", str(batch_rows), str(stream_rows), batch_rows == stream_rows),
     ("revenue", f"{batch_rev:,.2f}", f"{stream_rev:,.2f}", abs(batch_rev - stream_rev) < 0.01)],
    ["Metric", "Batch", "Streaming", "Match"],
))

## 6. Incrementality — the property that makes it streaming

Drop the held-back chunk into the landing directory and re-run both queries. Auto Loader
processes **only the new file**; the checkpoint remembers everything already seen. Re-running a
batch job over the same directory would reprocess all of it.

In [0]:
before = spark.table("cleaned_orders").count()

dbutils.fs.cp(HOLDBACK_CSV, f"{INBOX}/orders_part_{N_CHUNKS - 1:02d}.csv")
print(f"Dropped orders_part_{N_CHUNKS - 1:02d}.csv into the landing zone.")

(spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", SCHEMA_LOC)
      .option("header", True).option("encoding", "ISO-8859-1")
      .schema(bronze_schema).load(INBOX)
      .withColumn("source_file", F.col("_metadata.file_path"))
      .writeStream.option("checkpointLocation", CKPT_BRONZE)
      .outputMode("append").trigger(availableNow=True)
      .toTable("raw_orders")).awaitTermination()

(clean_orders(spark.readStream.table("raw_orders"))
      .writeStream.option("checkpointLocation", CKPT_SILVER)
      .outputMode("append").trigger(availableNow=True)
      .toTable("cleaned_orders")).awaitTermination()

after = spark.table("cleaned_orders").count()
print(f"Rows before : {before:,}")
print(f"Rows after  : {after:,}")
print(f"Newly processed (only the new file): {after - before:,}")

display(
      spark.table("raw_orders")
      .groupBy("source_file").count().orderBy("source_file")
)

## 7. Windowed streaming aggregation with a watermark

Everything so far was stateless. This is the genuinely streaming-only concept: aggregating over
**event time** with a watermark bounding how long state is kept.

⚠️ **The gotcha to know:** with `outputMode("append")`, a window is emitted only once the
watermark has passed its end — so with `availableNow` the final windows often never appear, and
beginners conclude the query is broken. `complete` mode to a memory sink sidesteps that for
demonstration. In production you would use append + Delta and accept the emission delay.

In [0]:
CKPT_WINDOW = f"{VOLUME}/_ckpt_window"
dbutils.fs.rm(CKPT_WINDOW, recurse=True)

windowed = (
    spark.readStream.table("cleaned_orders")
    .filter(F.col("is_valid_sale"))
    .withWatermark("InvoiceTimestamp", "7 days")
    .groupBy(
        F.window("InvoiceTimestamp", "30 days"),
        F.col("Country")
    )
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("Revenue"),
        # countDistinct is NOT supported in streaming aggregations.
        # approx_count_distinct uses HyperLogLog and is suitable for streaming.
        F.approx_count_distinct("InvoiceNo").alias("Orders_Approx")
    )
)

q_win = (
    windowed.writeStream
    .format("memory")
    .queryName("windowed_revenue")
    .option("checkpointLocation", CKPT_WINDOW)
    .outputMode("complete")
    .trigger(availableNow=True)
    .start()
)

q_win.awaitTermination()

display(
    spark.sql("""
        SELECT
            window.start AS Window_Start,
            window.end AS Window_End,
            Country,
            Revenue,
            Orders_Approx
        FROM windowed_revenue
        ORDER BY Window_Start, Revenue DESC
    """)
)

---
# 8. Gold — KPIs

Gold reads the Silver Delta table in batch. That is deliberate and standard: aggregations that
need `countDistinct`, window functions, or full-history recomputation are far cheaper and simpler
as batch over an incrementally-maintained Silver table than as streaming state.

In [0]:
silver = spark.table("cleaned_orders") 
# No .cache() — serverless compute does not support PERSIST and manages 
# its own disk caching, so an explicit cache call both fails and provides no benefit. 
TOTAL_LINES = silver.count() 
valid = silver.filter(F.col("is_valid_sale")) 
print(f"Total lines: {TOTAL_LINES:,} Valid sale lines: {valid.count():,}")

## KPI 1 — Revenue by Country

In [0]:
kpi_revenue_by_country = (
    valid
    .filter(F.col("Country").isNotNull())
    .groupBy("Country")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("Revenue"),
        F.countDistinct("InvoiceNo").alias("Orders"),
        F.countDistinct("CustomerID").alias("Customers"),
    )
    .withColumn(
        "Revenue_Per_Order",
        F.round(F.col("Revenue") / F.col("Orders"), 2)
    )
    .orderBy(F.desc("Revenue"))
)

display(kpi_revenue_by_country)

## KPI 2 — Top Products by Revenue

Non-product stock codes are excluded here — postage is revenue, but it is not a *product*, and
leaving it in makes `POSTAGE` your best seller.

In [0]:
kpi_top_products = (
    valid
    .filter(
        ~F.col("is_non_product")
        & F.col("Description").isNotNull()
    )
    .groupBy("StockCode", "Description")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("Revenue"),
        F.sum("Quantity").alias("Units_Sold"),
    )
    .orderBy(F.desc("Revenue"))
)

display(kpi_top_products.limit(20))

## KPI 3 — Monthly Revenue Trend

In [0]:
kpi_monthly_revenue = (
    valid
    .filter(F.col("year_month").isNotNull())
    .groupBy("year_month")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("Revenue"),
        F.countDistinct("InvoiceNo").alias("Orders"),
    )
    .orderBy("year_month")
)

w = Window.orderBy("year_month")

kpi_monthly_revenue = kpi_monthly_revenue.withColumn(
    "MoM_Growth_Pct",
    F.round(
        (
            (F.col("Revenue") - F.lag("Revenue").over(w))
            / F.lag("Revenue").over(w)
            * 100
        ),
        2,
    ),
)

display(kpi_monthly_revenue)

## KPI 4 — Return Rate and Cancellation Rate

Two different things, often conflated. A cancellation voids the whole invoice; a return is a
negative line against a delivered order.

In [0]:
return_lines = silver.filter(F.col("is_return")).count()
cancel_lines = silver.filter(F.col("is_cancellation")).count()

returned_value = (
    silver
    .filter(F.col("is_return"))
    .agg(F.sum("line_revenue"))
    .first()[0]
    or 0.0
)

gross_revenue = (
    valid
    .agg(F.sum("line_revenue"))
    .first()[0]
    or 0.0
)

kpi_returns = spark.createDataFrame(
    [(
        TOTAL_LINES,
        return_lines,
        round(return_lines / TOTAL_LINES * 100, 2)
        if TOTAL_LINES else None,
        cancel_lines,
        round(cancel_lines / TOTAL_LINES * 100, 2)
        if TOTAL_LINES else None,
        round(gross_revenue, 2),
        round(abs(returned_value), 2),
        round(abs(returned_value) / gross_revenue * 100, 2)
        if gross_revenue else None,
    )],
    [
        "Total_Lines",
        "Return_Lines",
        "Return_Rate_Pct",
        "Cancel_Lines",
        "Cancel_Rate_Pct",
        "Gross_Revenue",
        "Returned_Value",
        "Returned_Value_Pct",
    ],
)

display(kpi_returns)

## KPI 5 — Average Order Value

Computed per *invoice*, not per line — averaging line revenue would answer a different and far
less useful question.

In [0]:
order_totals = (
    valid
    .groupBy("InvoiceNo")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("Order_Value"),
        F.sum("Quantity").alias("Items"),
        F.first("Country").alias("Country"),
    )
)

kpi_aov = order_totals.agg(
    F.count("*").alias("Orders"),
    F.round(F.avg("Order_Value"), 2).alias("AOV"),
    F.round(
        F.expr("percentile_approx(Order_Value, 0.5)"),
        2
    ).alias("Median_Order_Value"),
    F.round(F.avg("Items"), 1).alias("Avg_Items_Per_Order"),
    F.round(F.max("Order_Value"), 2).alias("Largest_Order"),
)

display(kpi_aov)

# Top 10 largest orders
display(
    order_totals
    .orderBy(F.desc("Order_Value"))
    .limit(10)
)

## KPI 6 — Repeat Customer Rate

Guest checkouts (null `CustomerID`) are ~25% of rows and **cannot** be attributed. They are
excluded from the denominator and reported separately — folding them in either way would quietly
bias the rate.

In [0]:
identified = valid.filter(F.col("CustomerID").isNotNull())

customer_orders = (
    identified
    .groupBy("CustomerID")
    .agg(
        F.countDistinct("InvoiceNo").alias("Orders"),
        F.round(F.sum("line_revenue"), 2).alias("Lifetime_Value"),
    )
)

total_customers = customer_orders.count()

repeat_customers = (
    customer_orders
    .filter(F.col("Orders") > 1)
    .count()
)

guest_lines = (
    valid
    .filter(F.col("is_guest_checkout"))
    .count()
)

avg_lifetime_value = (
    customer_orders
    .agg(F.avg("Lifetime_Value"))
    .first()[0]
    or 0.0
)

valid_count = valid.count()

kpi_repeat = spark.createDataFrame(
    [(
        total_customers,
        repeat_customers,
        round(repeat_customers / total_customers * 100, 2)
        if total_customers else None,
        round(avg_lifetime_value, 2),
        guest_lines,
        round(guest_lines / valid_count * 100, 2)
        if valid_count else None,
    )],
    [
        "Identified_Customers",
        "Repeat_Customers",
        "Repeat_Rate_Pct",
        "Avg_Lifetime_Value",
        "Guest_Checkout_Lines",
        "Guest_Lines_Pct",
    ],
)

display(kpi_repeat)

# Top 10 customers by lifetime value
display(
    customer_orders
    .orderBy(F.desc("Lifetime_Value"))
    .limit(10)
)

## KPI 7 — Peak Trading Hours

In [0]:
kpi_peak_hours = (
    valid
    .filter(F.col("invoice_hour").isNotNull())
    .groupBy("invoice_hour")
    .agg(
        F.round(F.sum("line_revenue"), 2).alias("Revenue"),
        F.countDistinct("InvoiceNo").alias("Orders"),
    )
    .orderBy("invoice_hour")
)

display(kpi_peak_hours)

busiest_hour = (
    kpi_peak_hours
    .orderBy(F.desc("Revenue"))
    .first()
)

print("Busiest hour:", busiest_hour)

## KPI 8 — Data Quality Scorecard

Every flag set in `clean_orders` rolled up into one table. This is the KPI that tells you whether
to trust the other seven.

In [0]:
FLAGS = [
    "is_cancellation",
    "is_return",
    "is_non_product",
    "is_guest_checkout",
    "has_invalid_price",
    "has_missing_description",
    "has_unparseable_date",
]

counts = (
    silver
    .select(
        [
            F.sum(F.col(c).cast("int")).alias(c)
            for c in FLAGS
        ]
    )
    .first()
    .asDict()
)

kpi_quality = (
    spark.createDataFrame(
        [
            (
                c,
                int(n or 0),
                round((n or 0) / TOTAL_LINES * 100, 2)
                if TOTAL_LINES else 0.0,
            )
            for c, n in counts.items()
        ],
        [
            "Quality_Flag",
            "Row_Count",
            "Pct_Of_Rows",
        ],
    )
    .orderBy(F.desc("Pct_Of_Rows"))
)

display(kpi_quality)

usable_pct = (
    round(valid.count() / TOTAL_LINES * 100, 2)
    if TOTAL_LINES
    else 0.0
)

print(
    f"Usable-for-revenue rows: "
    f"{usable_pct}% of {TOTAL_LINES:,}"
)

---
## 9. Summary

In [0]:
summary = [
    ("Data source",              "SYNTHETIC (generated)" if USING_SYNTHETIC else "UCI Online Retail"),
    ("Bronze rows ingested",     f"{spark.table('raw_orders').count():,}"),
    ("Silver rows cleaned",      f"{TOTAL_LINES:,}"),
    ("Usable for revenue",       f"{usable_pct}%"),
    ("Gross revenue",            f"{gross_revenue:,.2f}"),
    ("Orders",                   f"{order_totals.count():,}"),
    ("Average order value",      f"{kpi_aov.first()['AOV']:,.2f}"),
    ("Top country",              kpi_revenue_by_country.first()["Country"]),
    ("Top product",              kpi_top_products.first()["Description"]),
    ("Return rate",              f"{round(return_lines / TOTAL_LINES * 100, 2)}%"),
    ("Repeat customer rate",     f"{round(repeat_customers / total_customers * 100, 2)}%"),
]
display(spark.createDataFrame(summary, ["Metric", "Value"]))

## 10. Cleanup

Stop any lingering streams and release the cache. Skip the `DROP SCHEMA` unless you want to
re-run the whole notebook from scratch.

In [0]:
for q in spark.streams.active:
      print("Stopping:", q.name)
      q.stop()

  # Full reset — uncomment to start over:
  # spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")

print("Done.")


---
## What this exercise demonstrates

| Concept | Where |
|---|---|
| Auto Loader incremental file ingestion | §4, §6 |
| Explicit schema over inference on dirty CSV | §4 |
| Checkpointing and exactly-once file tracking | §4, §5, §6 |
| Medallion architecture (bronze/silver/gold) | throughout |
| Engine-agnostic transform logic | §3, §5 |
| Batch/stream parity verification | §5 |
| Event-time windowing with watermarks | §7 |
| Append-vs-complete output-mode trade-off | §7 |
| Classifying dirt rather than deleting it | §3, §4 |
| Quality flags as first-class columns | §3, KPI 8 |

### Things I would do differently in production
- Land raw files in object storage with lifecycle rules, not a volume.
- Use Delta Live Tables / Lakeflow declarative pipelines for the bronze→silver hop, so
  expectations and quarantining are declarative instead of hand-rolled flags.
- Add `OPTIMIZE` / `ZORDER BY (invoice_date, Country)` on Silver once it is large.
- Replace `trigger(availableNow=True)` with a continuous or scheduled trigger.